In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# 1. Chỉ định rõ model BERT để khớp với token [MASK]
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 2. Câu đầu vào
input_sentence = "Hanoi is the [MASK] of Vietnam."

# 3. Thực hiện dự đoán
predictions = mask_filler(input_sentence, top_k=5)

# 4. In kết quả
print(f"Mô hình đang sử dụng: {mask_filler.model.config._name_or_path}")
print(f"Câu gốc: {input_sentence}\n" + "-"*30)
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' | Độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


Mô hình đang sử dụng: bert-base-uncased
Câu gốc: Hanoi is the [MASK] of Vietnam.
------------------------------
Dự đoán: 'capital' | Độ tin cậy: 0.9991
 -> Câu hoàn chỉnh: hanoi is the capital of vietnam.
Dự đoán: 'center' | Độ tin cậy: 0.0001
 -> Câu hoàn chỉnh: hanoi is the center of vietnam.
Dự đoán: 'birthplace' | Độ tin cậy: 0.0001
 -> Câu hoàn chỉnh: hanoi is the birthplace of vietnam.
Dự đoán: 'headquarters' | Độ tin cậy: 0.0001
 -> Câu hoàn chỉnh: hanoi is the headquarters of vietnam.
Dự đoán: 'city' | Độ tin cậy: 0.0001
 -> Câu hoàn chỉnh: hanoi is the city of vietnam.


In [ ]:
from transformers import pipeline

# 1. Tải pipeline text-generation với mô hình GPT-2
generator = pipeline("text-generation", model="gpt2")

# 2. Câu mồi
prompt = "The best thing about learning NLP is"

# 3. Sinh văn bản
# max_length: độ dài tối đa bao gồm cả câu mồi
generated_texts = generator(prompt, max_length=50, num_return_sequences=1, pad_token_id=50256)

# 4. In kết quả
print(f"Câu mồi: '{prompt}'\n" + "-"*30)
print("Văn bản được sinh ra:")
print(generated_texts[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Câu mồi: 'The best thing about learning NLP is'
------------------------------
Văn bản được sinh ra:
The best thing about learning NLP is that it feels like a real tool, not some kind of "magic" that you can do from scratch.

"You can build a great language with NLP and you can build a great language without NLP. That's just not possible."

That's what I found in the course. It's the best thing about learning NLP is that it feels like a real tool, not some kind of "magic" that you can do from scratch. I believe that NLP is about learning how to code at the level I expect you to.

By doing this, you are also giving yourself the training and the tools to get a better understanding of what you are doing.

It's not just about learning how to use NLP. It's also about what you are doing in the real world.

"The great thing about learning NLP is that it feels like a real tool, not some kind of "magic" that you can do from scratch."

In other words, using NLP is about learning how to code at t

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Khởi tạo Tokenizer và Model BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Câu đầu vào
sentences = ["This is a sample sentence."]

# 3. Tokenize câu (chuyển chữ thành số)
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# 4. Đưa qua mô hình (không tính gradient để tiết kiệm RAM)
with torch.no_grad():
    outputs = model(**inputs)

# 5. Thực hiện Mean Pooling
# last_hidden_state chứa vector của từng từ một
last_hidden_state = outputs.last_hidden_state
attention_mask = inputs['attention_mask']

# Mở rộng mask để có cùng kích thước với hidden states
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

# Tính tổng các vector nhân với mask (để bỏ qua padding) và chia trung bình
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask

# 6. In kết quả
print("Kích thước vector biểu diễn (Embedding):", sentence_embedding.shape)
print("\n10 giá trị đầu tiên của vector:")
print(sentence_embedding[0][:10])

Kích thước vector biểu diễn (Embedding): torch.Size([1, 768])

10 giá trị đầu tiên của vector:
tensor([-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5784e-02,
        -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0647e-05, -7.4273e-02])
